# Phase 5: Promotion Effectiveness Synthesis

**Objective:**
To determine which customer segments (Champions, Loyal, At Risk, Lost) are consuming the promotional discounts. 

**Hypothesis:**
If promotional revenue is heavily concentrated in the "Champions" segment, the promotions are highly inefficient—they are subsidizing loyal customers rather than acquiring or reactivating at-risk ones.

In [1]:
import pandas as pd
import numpy as np
import datetime as dt

print("Synthesizing Promo Effectiveness...")
df = pd.read_csv("D:/Retail Pricing & Promotion Analytics/data/processed/cleaned_online_retail.csv")
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)

# 1. Rebuild Promo Flag (Item-Month Level)
item_avg = df.groupby('StockCode')['UnitPrice'].mean().reset_index(name='AllTimeAvg')
monthly_item = df.groupby(['YearMonth', 'StockCode']).agg(MonthlyAvg=('UnitPrice', 'mean')).reset_index()
promo_df = pd.merge(monthly_item, item_avg, on='StockCode')
promo_df['IsPromo'] = np.where(promo_df['MonthlyAvg'] < (promo_df['AllTimeAvg'] * 0.85), 1, 0)

# Merge promo flag back to main transactions
df = pd.merge(df, promo_df[['YearMonth', 'StockCode', 'IsPromo']], on=['YearMonth', 'StockCode'], how='left')

# 2. Rebuild RFM Tiers
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'Revenue': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])
rfm['RFM_Score'] = rfm[['R_Score', 'F_Score', 'M_Score']].sum(axis=1)

def get_tier(score):
    if score >= 13: return 'Champions'
    elif score >= 10: return 'Loyal'
    elif score >= 6: return 'At Risk'
    else: return 'Lost'
    
rfm['Customer_Tier'] = rfm['RFM_Score'].apply(get_tier)

# Merge tiers back to main transactions
df = pd.merge(df, rfm[['CustomerID', 'Customer_Tier']], on='CustomerID', how='inner')

# 3. Calculate Promo Revenue by Segment
promo_transactions = df[df['IsPromo'] == 1]
total_promo_revenue = promo_transactions['Revenue'].sum()

tier_promo_revenue = promo_transactions.groupby('Customer_Tier')['Revenue'].sum().reset_index()
tier_promo_revenue['Pct_of_Promo_Rev'] = (tier_promo_revenue['Revenue'] / total_promo_revenue) * 100
tier_promo_revenue = tier_promo_revenue.sort_values('Pct_of_Promo_Rev', ascending=False)

print("\n--- Promotional Revenue Distribution by Tier ---")
print(tier_promo_revenue.to_string(index=False))

champions_pct = tier_promo_revenue[tier_promo_revenue['Customer_Tier'] == 'Champions']['Pct_of_Promo_Rev'].values[0]

print(f"\nFINAL INSIGHT:")
print(f"{champions_pct:.1f}% of promotional revenue came from customers already in the top RFM tier - meaning promotions are subsidizing loyal customers rather than reactivating at-risk ones.")

Synthesizing Promo Effectiveness...

--- Promotional Revenue Distribution by Tier ---
Customer_Tier    Revenue  Pct_of_Promo_Rev
    Champions 340316.460         67.210442
        Loyal 104643.141         20.666387
      At Risk  55155.330         10.892844
         Lost   6229.690          1.230326

FINAL INSIGHT:
67.2% of promotional revenue came from customers already in the top RFM tier - meaning promotions are subsidizing loyal customers rather than reactivating at-risk ones.
